# 03. Validar contra a coorte

O modelo do [`02_deduplicar_splink.ipynb`](02_deduplicar_splink.ipynb) é treinado
sem ver a coorte. Aqui ela entra só como ground truth, numa *labels table* do
Splink com duas partes:

- **positivos**: pares Censo ↔ CPF que a coorte diz ser a mesma pessoa;
- **negativos difíceis**: pares de indivíduos *distintos* da coorte que passam
  pelas mesmas blocking rules do modelo (mesmo nome, mesmo sobrenome ou mesma
  data de nascimento).

Os negativos explícitos são o ponto central. Rotulando só os positivos, todo par
não rotulado vira não-match por omissão e a precision cai artificialmente.

**Pré-requisito:** NB02 executado (`splink_model.json` e `splink_clusters.parquet`).

In [1]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from IPython.display import display

from config import (
    COHORT_DEDUP_ARQUIVO,
    METRICAS_COHORT,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    TABELA_LIMPA,
    cpf_norm_sql,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

print_paths()
require_input(COHORT_DEDUP_ARQUIVO, label='COHORT')
require_input(SPLINK_MODEL_JSON, label='SPLINK_MODEL_JSON (rode o NB02 antes)')
require_input(SPLINK_CLUSTERS, label='SPLINK_CLUSTERS (rode o NB02 antes)')

con = get_connection()
require_tables(con, [TABELA_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

con.execute(f"""
CREATE OR REPLACE TABLE cohort_dedup_raw AS
SELECT * FROM read_parquet('{COHORT_DEDUP_ARQUIVO}')
""")
print('Registros na coorte:', con.execute('SELECT COUNT(*) FROM cohort_dedup_raw').fetchone()[0])

OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_pessoas_2022_20260505.parquet
CENSO_CEP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/raw/censo/data_cep_uniq.csv
COHORT_DEDUP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/capefe/dados/CohortDados/cohort_dedup.parquet
FILTRO_UF: None
FILTRO_MUNICIPIO: 2111300
USE_PHONETIC_STRIP_VOWELS: False
ANO_OBITO_CORTE: 2021
ANO_NASCIMENTO_MIN: 1900
SEXO_VALIDOS: ('M', 'F')
DUCKDB_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/probabilistico.duckdb
splink_input → registro_limpo


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Registros na coorte: 53790030


## 1. Checagem 1:1 da coorte

Um `person_id_censo` ligado a dois CPFs (ou o inverso) não serve como ground
truth: não dá para dizer qual dos dois é o par correto, e usar ambos rotularia
um par verdadeiro como erro. Só os pares estritamente 1:1 seguem adiante.

In [2]:
CPF_GT = cpf_norm_sql('CPF_NORM')
con.execute(f'''
CREATE OR REPLACE TABLE cohort_pares AS
SELECT DISTINCT
    CAST(PERSON_ID_CENSO AS VARCHAR) AS person_id_censo,
    {CPF_GT} AS cpf_norm
FROM cohort_dedup_raw
WHERE PERSON_ID_CENSO IS NOT NULL AND CPF_NORM IS NOT NULL
''')

con.execute('''
CREATE OR REPLACE TABLE cohort_pares_card AS
SELECT *,
    COUNT(*) OVER (PARTITION BY person_id_censo) AS n_cpf_por_censo,
    COUNT(*) OVER (PARTITION BY cpf_norm) AS n_censo_por_cpf
FROM cohort_pares
''')

display(con.execute('''
SELECT
    COUNT(*) AS n_pares,
    COUNT(DISTINCT person_id_censo) AS n_censo,
    COUNT(DISTINCT cpf_norm) AS n_cpf,
    SUM(CASE WHEN n_cpf_por_censo > 1 THEN 1 ELSE 0 END) AS pares_censo_ambiguo,
    SUM(CASE WHEN n_censo_por_cpf > 1 THEN 1 ELSE 0 END) AS pares_cpf_ambiguo
FROM cohort_pares_card
''').df())

con.execute('''
CREATE OR REPLACE TABLE ground_truth_pairs AS
SELECT
    'censo_' || person_id_censo AS unique_id_censo,
    'cpf_' || cpf_norm AS unique_id_cpf,
    person_id_censo,
    cpf_norm
FROM cohort_pares_card
WHERE n_cpf_por_censo = 1 AND n_censo_por_cpf = 1
''')
print('Pares 1:1 utilizáveis:', con.execute('SELECT COUNT(*) FROM ground_truth_pairs').fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_pares,n_censo,n_cpf,pares_censo_ambiguo,pares_cpf_ambiguo
0,53790030,53776799,53790030,26462.0,0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pares 1:1 utilizáveis: 53763568


## 2. Cobertura no subset

A coorte é nacional; `registro_limpo` pode estar filtrado por UF/município e
já perdeu quem morreu antes do Censo (NB00b).
Só os pares cujos dois lados existem no subset podem ser avaliados.

In [3]:
con.execute(f'''
CREATE OR REPLACE TABLE gt_no_subset AS
SELECT gt.*
FROM ground_truth_pairs gt
JOIN {SPLINK_INPUT_VIEW} c ON c.unique_id = gt.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} p ON p.unique_id = gt.unique_id_cpf
''')

display(con.execute('''
SELECT
    (SELECT COUNT(*) FROM ground_truth_pairs) AS n_pares_1a1,
    (SELECT COUNT(*) FROM gt_no_subset) AS n_pares_no_subset,
    ROUND(
        100.0 * (SELECT COUNT(*) FROM gt_no_subset)
        / NULLIF((SELECT COUNT(*) FROM ground_truth_pairs), 0),
        2
    ) AS pct_cobertura
''').df())

n_gt = con.execute('SELECT COUNT(*) FROM gt_no_subset').fetchone()[0]
if n_gt == 0:
    raise RuntimeError(
        'Nenhum par da coorte caiu no subset — confira o filtro geográfico do NB00.'
    )

,n_pares_1a1,n_pares_no_subset,pct_cobertura
0,53763568,211143,0.39


## 3. Labels table: positivos + negativos difíceis

Os negativos usam as mesmas chaves das blocking rules do NB02. Pares que não
passam por nenhuma blocking rule nunca são pontuados pelo modelo, então rotulá-los
como negativos só infla artificialmente os verdadeiros negativos.

`clerical_match_score` é o rótulo humano: 1.0 para o mesmo indivíduo, 0.0 para
indivíduos distintos.

In [4]:
N_NEG_POR_ANCORA = 5

con.execute(f'''
CREATE OR REPLACE TABLE gt_registros AS
SELECT
    gt.unique_id_censo, gt.unique_id_cpf, gt.person_id_censo,
    c.primeiro_nome, c.ultimo_nome, c.data_nascimento
FROM gt_no_subset gt
JOIN {SPLINK_INPUT_VIEW} c ON c.unique_id = gt.unique_id_censo
''')

con.execute(f'''
CREATE OR REPLACE TABLE labels_negativos AS
SELECT unique_id_l, unique_id_r FROM (
    SELECT
        least(a.unique_id_censo, b.unique_id_cpf) AS unique_id_l,
        greatest(a.unique_id_censo, b.unique_id_cpf) AS unique_id_r,
        row_number() OVER (PARTITION BY a.unique_id_censo ORDER BY random()) AS rn
    FROM gt_registros a
    JOIN gt_registros b
      ON a.person_id_censo <> b.person_id_censo
     AND (
        (a.primeiro_nome = b.primeiro_nome AND a.ultimo_nome = b.ultimo_nome)
        OR (a.ultimo_nome = b.ultimo_nome AND a.data_nascimento = b.data_nascimento)
        OR (a.primeiro_nome = b.primeiro_nome AND a.data_nascimento = b.data_nascimento)
     )
)
WHERE rn <= {N_NEG_POR_ANCORA}
''')

con.execute('''
CREATE OR REPLACE TABLE splink_labels AS
SELECT
    least(unique_id_censo, unique_id_cpf) AS unique_id_l,
    greatest(unique_id_censo, unique_id_cpf) AS unique_id_r,
    1.0 AS clerical_match_score
FROM gt_no_subset
UNION
SELECT unique_id_l, unique_id_r, 0.0 AS clerical_match_score
FROM labels_negativos
''')

display(con.execute('''
SELECT clerical_match_score, COUNT(*) AS n
FROM splink_labels GROUP BY 1 ORDER BY 1 DESC
''').df())

n_neg = con.execute('SELECT COUNT(*) FROM labels_negativos').fetchone()[0]
if n_neg == 0:
    print(
        'AVISO: nenhum negativo difícil gerado. Com poucos pares no subset a '
        'precision fica sem base de comparação — amplie o recorte geográfico.'
    )
df_labels = con.execute('SELECT * FROM splink_labels').df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clerical_match_score,n
0,1.0,211143
1,0.0,453299


## 4. Carregar o modelo treinado

O `Linker` é reconstruído a partir do JSON salvo pelo NB02 — nenhum parâmetro é
reestimado aqui, então a coorte não influencia o modelo.

In [12]:
import json
# 1. Load the existing JSON data from a file
with open(SPLINK_MODEL_JSON, "r") as file:
    data = json.load(file)

# 2. Add the new key-value pair (it automatically goes to the end)
data["retain_intermediate_calculation_columns"] = True
data

{'link_type': 'dedupe_only',
 'probability_two_random_records_match': 2.1404364100231013e-07,
 'retain_matching_columns': True,
 'retain_intermediate_calculation_columns': True,
 'additional_columns_to_retain': [],
 'sql_dialect': 'duckdb',
 'linker_uid': 'm0gsc44q',
 'em_convergence': 0.0001,
 'max_iterations': 25,
 'bayes_factor_column_prefix': 'bf_',
 'term_frequency_adjustment_column_prefix': 'tf_',
 'comparison_vector_value_column_prefix': 'gamma_',
 'unique_id_column_name': 'unique_id',
 'source_dataset_column_name': 'source_dataset',
 'blocking_rules_to_generate_predictions': [{'blocking_rule': '(l."primeiro_nome" = r."primeiro_nome") AND (l."ultimo_nome" = r."ultimo_nome")',
   'sql_dialect': 'duckdb'},
  {'blocking_rule': '(l."ultimo_nome" = r."ultimo_nome") AND (l."data_nascimento" = r."data_nascimento")',
   'sql_dialect': 'duckdb'},
  {'blocking_rule': '(l."primeiro_nome" = r."primeiro_nome") AND (l."data_nascimento" = r."data_nascimento")',
   'sql_dialect': 'duckdb'},
  {

In [13]:
from splink import Linker


linker = Linker(SPLINK_INPUT_VIEW, data, db_api=db_api)
labels_sdf = linker.table_management.register_labels_table(df_labels, overwrite=True)
print(f'Labels registradas: {len(df_labels):,}')

Labels registradas: 664,442


## 5. Accuracy por threshold

Curva de precision, recall e F1 sobre a labels table. Use-a para escolher o
threshold de produção.

In [14]:
linker.evaluation.accuracy_analysis_from_labels_table(
    labels_sdf,
    output_type='accuracy',
    match_weight_round_to_nearest=0.02,
)

alt.LayerChart(...)

## 6. Erros de predição

Waterfall dos pares em que o modelo discorda da coorte, no threshold escolhido.
Falsos negativos que não passaram por nenhuma blocking rule não aparecem aqui —
esses são perda de blocking, não de scoring, e são medidos na seção 7.

In [15]:
THRESHOLD_AVALIACAO = 0.95

records_fp = linker.evaluation.prediction_errors_from_labels_table(
    labels_sdf,
    threshold_match_probability=THRESHOLD_AVALIACAO,
    include_false_negatives=False,
    include_false_positives=True,
).as_record_dict(limit=20)
print('Falsos positivos (amostra):', len(records_fp))
if records_fp:
    display(linker.visualisations.waterfall_chart(records_fp, filter_nulls=False))

Falsos positivos (amostra): 20


alt.LayerChart(...)

In [16]:
records_fn = linker.evaluation.prediction_errors_from_labels_table(
    labels_sdf,
    threshold_match_probability=THRESHOLD_AVALIACAO,
    include_false_negatives=True,
    include_false_positives=False,
).as_record_dict(limit=20)
print('Falsos negativos (amostra):', len(records_fn))
if records_fn:
    display(linker.visualisations.waterfall_chart(records_fn, filter_nulls=False))

Falsos negativos (amostra): 20


alt.LayerChart(...)

## 7. Recall cross-source nos clusters

Métrica de ponta a ponta: dos pares Censo ↔ CPF da coorte, quantos caíram no
mesmo cluster do NB02. Diferente da seção 6, isto inclui as perdas de blocking.

In [17]:
con.execute(f'''
CREATE OR REPLACE TABLE splink_clusters AS
SELECT * FROM read_parquet('{SPLINK_CLUSTERS}')
''')

metricas = con.execute('''
SELECT
    COUNT(*) AS n_pares_gt,
    SUM(CASE WHEN cl.cluster_id = cr.cluster_id THEN 1 ELSE 0 END) AS hits,
    ROUND(
        1.0 * SUM(CASE WHEN cl.cluster_id = cr.cluster_id THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        4
    ) AS recall_cross_source
FROM gt_no_subset gt
LEFT JOIN splink_clusters cl ON cl.unique_id = gt.unique_id_censo
LEFT JOIN splink_clusters cr ON cr.unique_id = gt.unique_id_cpf
''').df()
display(metricas)

metricas['threshold_avaliacao'] = THRESHOLD_AVALIACAO
metricas['n_labels_positivos'] = int((df_labels['clerical_match_score'] == 1.0).sum())
metricas['n_labels_negativos'] = int((df_labels['clerical_match_score'] == 0.0).sum())
metricas.to_csv(METRICAS_COHORT, index=False)
print('Métricas salvas:', METRICAS_COHORT)

con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_pares_gt,hits,recall_cross_source
0,211143,192073.0,0.9097


Métricas salvas: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/metricas_cohort.csv
